# Three-Stage Sequential Sigma Fitting

Fits the three-regime noise parameters **independently** using toy
experiments at the relevant ISI ranges:

| Stage | Parameter | Toy ISIs | Rationale |
|-------|-----------|----------|-----------|
| A | `sigma0` | 0 | Only 1 step of noise — only sigma0 matters |
| B | `sigma1` | 1, 2, 4 | sigma0 fixed from Stage A; sigma1 is the only free variable |
| C | `sigma2` | 8, 16, 32, 64 | sigma0 & sigma1 fixed; sigma2 is free |

Each stage does a cheap **1-D grid search** with iterative refinement,
then the final fitted parameters are evaluated on the full experiment.

In [ ]:
# ===================== Imports =====================
import sys, os, glob, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import pdist

# ---- project paths (adjust to your environment) ----
sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import (
    run_experiment_scores,
    make_noise_schedule,
)
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

# ---- new modular fitting code ----
from utls.toy_experiments import (
    make_isi_n_block_experiment,
    make_toy_experiment_list,
    make_multi_isi_toy_experiments,
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    three_stage_fit,
    plot_sigma_fit,
)

## 1. Load config & data

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


CONFIG_PATH = (
    "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
# ---- experiment ----
exp_cfg = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)

isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
t_step = noise_cfg["t_step"]
noise_tag = f"{noise_mode}_t{t_step}"

# ---- representation ----
repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

# ---- load human data ----
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)

## 2. Build encoder & encode stimuli

In [ ]:
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = (
    "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,
    model_name=encoder_type,
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

## 3. Set up parameter bounds

In [ ]:
from scipy.spatial.distance import pdist


def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))


d50 = median_pairwise_distance(X_np, metric="cosine")
print(f"Median pairwise cosine distance: {d50:.6f}")

param_bounds = {
    "sigma0": (
        noise_cfg["sigma0_min"],
        noise_cfg["sigma0_max"],
    ),
    "sigma1": (
        noise_cfg["sigma1_min"] * d50,
        noise_cfg["sigma1_max"] * d50,
    ),
    "sigma2": (
        noise_cfg["sigma2_min"] * d50,
        noise_cfg["sigma2_max"] * d50,
    ),
}

print("Parameter bounds:")
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

## 4. Stimulus pool for toy experiments

In [ ]:
# collect all unique stimuli across experiment sequences
stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"Stimulus pool size: {len(stimulus_pool)}")

# sanity check: for ISI=64 we need blocks of 65, so pool should be >= 65
assert len(stimulus_pool) >= 65, (
    f"Stimulus pool ({len(stimulus_pool)}) too small for ISI-64 blocks (need >= 65)"
)

## 5. Run three-stage fit

This is the main call.  It:
1. Generates ISI-specific toy experiments automatically
2. Fits sigma0 → sigma1 → sigma2 sequentially
3. Produces diagnostic plots per stage

In [ ]:
fit_result = three_stage_fit(
    run_experiment_fn=run_experiment_scores,
    param_bounds=param_bounds,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    stimulus_pool=stimulus_pool,
    human_curve=human_curve,
    isis=isis,
    t_step=t_step,
    # ISI groupings (defaults are fine, but explicit for clarity)
    isi_sigma0=[0],
    isi_sigma1=[1, 2, 4],
    isi_sigma2=[8, 16, 32, 64],
    # toy experiment settings
    n_experiments_per_isi=20,
    k_stimuli_per_exp=10,     # adjusted upward automatically for large ISIs
    # grid search settings
    n_grid=15,
    n_mc=32,
    n_refine_iters=2,
    seed=0,
    verbose=True,
    plot=True,
)

print("\nFitted parameters:")
print(f"  sigma0 = {fit_result['sigma0']:.6f}")
print(f"  sigma1 = {fit_result['sigma1']:.6f}")
print(f"  sigma2 = {fit_result['sigma2']:.6f}")

## 6. Final evaluation on full experiments

Now evaluate the fitted sigmas on the real experiments (all ISI conditions)
that all participants have taken.

In [ ]:
sigma0_fit = fit_result["sigma0"]
sigma1_fit = fit_result["sigma1"]
sigma2_fit = fit_result["sigma2"]

final_pb = {
    "sigma0": (sigma0_fit, sigma0_fit),
    "sigma1": (sigma1_fit, sigma1_fit),
    "sigma2": (sigma2_fit, sigma2_fit),
    "t_step": (t_step, t_step),
}

# re-compute human curve on full data (no subsampling)
human_curve_full = compute_human_curve(human_runs, is_multi, which_isi)

final_result = random_search_gridlike(
    n_samples=1,
    param_bounds=final_pb,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list,
    isis=isis,
    human_curve=human_curve_full,
    layer=encoder_name,
    encoder_name=encoder_name,
    hr_task_name=hr_task_name,
    subsample=len(exp_list),
    random_draw=False,
    seed=42,
)[0]

print("\nFinal evaluation (all ISIs):")
print(f"  Params: sigma0={sigma0_fit:.4f}, sigma1={sigma1_fit:.4f}, sigma2={sigma2_fit:.4f}")
print(f"  MSE per ISI: {final_result['mse_per_isi']}")
print(f"  MSE (ISI=0):  {final_result['mse_per_isi'][0]:.6f}")
print(f"  MSE (ISI>=1): {np.mean(final_result['mse_per_isi'][1:]):.6f}")

## 7. Summary plot: model vs human d' across all ISIs

In [ ]:
best_fits = generate_and_plot_model_decay_summary_v5(
    [final_result],
    human_curve_full,
    isis,
    metric_name="mse_per_isi",
    isi_indices=list(range(len(isis))),
    savedir=None,
    max_rows=1,
    hr_task_name=hr_task_name,
    encoder_name=encoder_name,
)

## 8. (Optional) Run stages individually for more control

If you want finer control over each stage (different grid sizes,
bounds, etc.), you can call `fit_sigma_1d` directly.

In [ ]:
# Example: re-fit sigma1 with wider bounds and more grid points
#
# from utls.sigma_fitting import fit_sigma_1d, plot_sigma_fit
# from utls.toy_experiments import make_multi_isi_toy_experiments
#
# sigma1_exps = make_multi_isi_toy_experiments(
#     stimulus_pool,
#     isi_values=[1, 2, 4],
#     n_experiments_per_isi=30,
#     k_stimuli=15,
#     seed=999,
# )
#
# sigma1_human = {1: human_curve[1], 2: human_curve[2], 4: human_curve[3]}
#
# stage_b_custom = fit_sigma_1d(
#     run_experiment_fn=run_experiment_scores,
#     sigma_name="sigma1",
#     sigma_bounds=(0.001, 5.0),          # wider bounds
#     fixed_sigmas={"sigma0": sigma0_fit, "sigma2": 0.1},
#     noise_mode=noise_mode,
#     metric=metric,
#     X0=X,
#     name_to_idx=name_to_idx,
#     experiments_by_isi=sigma1_exps,
#     human_dprimes_by_isi=sigma1_human,
#     t_step=t_step,
#     n_grid=20,                          # more grid points
#     n_mc=32,
#     n_refine_iters=3,                   # extra refinement
#     spacing="hybrid",
#     seed=123,
# )
#
# plot_sigma_fit(stage_b_custom, sigma1_human, title="Stage B (custom)")
# print(f"sigma1 = {stage_b_custom['best_sigma']:.6f}")